# Freezing the layers way

In [ ]:
# # -------------------------------------------------------------
# # 1️⃣ Install dependencies
# # -------------------------------------------------------------
# !pip install -q transformers datasets accelerate bitsandbytes

# # -------------------------------------------------------------
# # 2️⃣ Imports
# # -------------------------------------------------------------
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
# from datasets import load_dataset
# import os, re

# # -------------------------------------------------------------
# # 3️⃣ Load base model and tokenizer
# # -------------------------------------------------------------
# model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# # -------------------------------------------------------------
# # 4️⃣ Freeze everything
# # -------------------------------------------------------------
# for param in model.parameters():
#     param.requires_grad = False

# # -------------------------------------------------------------
# # 5️⃣ Unfreeze last 4 transformer blocks + lm_head
# # -------------------------------------------------------------
# for name, param in model.named_parameters():
#     if any(f"layers.{i}." in name for i in range(20, 24)):  # last 4 layers
#         param.requires_grad = True
#     if "lm_head" in name:
#         param.requires_grad = True

# # Verify how many parameters are trainable
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total = sum(p.numel() for p in model.parameters())
# print(f"✅ Trainable: {trainable/total*100:.2f}% of total parameters")

# # -------------------------------------------------------------
# # 6️⃣ Load your dataset
# # -------------------------------------------------------------
# dataset = load_dataset("json", data_files={"train": "pharma_non_instruction.jsonl"})

# def tokenize_fn(examples):
#     tokens = tokenizer(
#         examples["text"],
#         truncation=True,
#         padding="max_length",
#         max_length=512
#     )
#     tokens["labels"] = tokens["input_ids"].copy()
#     return tokens

# tokenized = dataset["train"].map(tokenize_fn, batched=True, remove_columns=["text"])

# # -------------------------------------------------------------
# # 7️⃣ Data collator for causal LM
# # -------------------------------------------------------------
# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# # -------------------------------------------------------------
# # 8️⃣ Training arguments
# # -------------------------------------------------------------
# os.environ["WANDB_DISABLED"] = "true"  # disable wandb
# training_args = TrainingArguments(
#     output_dir="./tinyllama-pharma-last4",
#     overwrite_output_dir=True,
#     num_train_epochs=2,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=1e-4,        # slightly higher since fewer params
#     fp16=True,
#     logging_steps=20,
#     save_steps=200,
#     save_total_limit=2,
#     report_to="none"
# )

# # -------------------------------------------------------------
# # 9️⃣ Trainer setup
# # -------------------------------------------------------------
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized,
#     data_collator=data_collator
# )

# # -------------------------------------------------------------
# # 🔟 Start training
# # -------------------------------------------------------------
# trainer.train()

# # -------------------------------------------------------------
# # 11️⃣ Save final model
# # -------------------------------------------------------------
# trainer.save_model("./tinyllama-pharma-last4-final")
# print("\n✅ Training completed and model saved!")